In [1]:
########################
#CODE INFORMATION

In [2]:
#Getting Data
# https://projectpythia.org/mrms-cookbook/notebooks/ch4-realtimedata/
# Authors:
# Ty Janoski
# City College of New York and NOAA/OAR National Severe Storms Laboratory
# Mya Sears
# NSF National Center for Atmospheric Research
# Bella Condo
# University at Albany (State University of New York)
# JD Heaton
# Metropolitan State University of Denver
# MaKenna Collins
# Jackson State University
# Maxwell Grover
# Argonne National Laboratory

In [3]:
########################
#LIBRARIES

In [4]:
#system packages
import os, re

#data loading packages
import pandas as pd

# Packages required to request and open data from AWS S3
import s3fs
import urllib
import tempfile
import gzip
import xarray as xr

In [270]:


# # Packages required for data visualization
# import datetime
# from datetime import timezone
# import numpy.ma as ma
# from metpy.plots import ctables
# import numpy as np
# import holoviews as hv
# import pandas as pd
# import panel as pn
# import hvplot.xarray 
# import matplotlib.colors as mcls
# from matplotlib.colors import Normalize

# hv.extension("bokeh")
# pn.extension()

In [8]:
########################
#DATA RETRIEVAL FUNCTIONS

In [9]:
def SubsetDataRegion(data, lat_range=(25, 35), lon_range=(-100, -90)):
    """
    Subset an xarray DataArray to a given lat/lon region.
    """
    # Convert longitudes to 0–360 range if dataset uses that convention
    if data.longitude.max() > 180:
        lon_min_target = 360 + lon_range[0] if lon_range[0] < 0 else lon_range[0]
        lon_max_target = 360 + lon_range[1] if lon_range[1] < 0 else lon_range[1]
    else:
        lon_min_target, lon_max_target = lon_range

    lat_min_target, lat_max_target = lat_range

    # Get nearest actual grid points
    lat_min = float(data.latitude.sel(latitude=lat_min_target, method='nearest'))
    lat_max = float(data.latitude.sel(latitude=lat_max_target, method='nearest'))
    lon_min = float(data.longitude.sel(longitude=lon_min_target, method='nearest'))
    lon_max = float(data.longitude.sel(longitude=lon_max_target, method='nearest'))

    # Apply slicing (note reversed latitude order if data is descending)
    subset = data.sel(latitude=slice(lat_max, lat_min),
                      longitude=slice(lon_min, lon_max))

    return subset

In [16]:
# ============================================================
# Helper: extract time from filename
# ============================================================
def extract_time_from_filename(fname):
    """Parse MRMS filename and return a pandas.Timestamp (UTC)."""
    match = re.search(r"_(\d{8}-\d{6})\.grib2\.gz", fname)
    return pd.to_datetime(match.group(1), format="%Y%m%d-%H%M%S", utc=True)

# ============================================================
# Select target times at 6-hour intervals
# ============================================================
def select_nearest_to_targets(files, interval_hours=6):
    """Return subset of file paths nearest to each N-hour UTC time."""
    # Extract actual timestamps from filenames
    file_times = pd.Series({f: extract_time_from_filename(f) for f in files}).sort_values()

    # Build list of desired UTC times covering file range
    start = file_times.min().floor(f"{interval_hours}h")   # lowercase h
    end   = file_times.max().ceil(f"{interval_hours}h")
    targets = pd.date_range(start, end, freq=f"{interval_hours}h", tz="UTC")

    selected = []
    for t in targets:
        nearest_idx = (abs(file_times - t)).argmin()
        selected.append(file_times.index[nearest_idx])

    return selected

def RetrieveMRMSRadarData(region,product,datestrings,
                          lat_range=(25, 35), lon_range=(-100, -90),
                          outputPath=""):
    aws = s3fs.S3FileSystem(anon=True)
    
    all_datasets = []
    for datestring in datestrings:
        print(f"\n=== Processing {datestring} ===")
    
        # List all files for this date
        try:
            data_files = aws.ls(f'noaa-mrms-pds/{region}/{product}/{datestring}/')
        except Exception as e:
            print(f"Could not access {datestring}: {e}")
            continue
    
        if not data_files:
            print(f"No files found for {datestring}")
            continue
    
        data_files = sorted(data_files)
    
        # (Optional) Filter to specific times or interval
        selected_files = select_nearest_to_targets(data_files, interval_hours=6)
    
        datasets = []
        for f in selected_files:
            print(f'working on {f}')
            try:
                response = urllib.request.urlopen(f"https://noaa-mrms-pds.s3.amazonaws.com/{f[14:]}")
                compressed_file = response.read()
    
                with tempfile.NamedTemporaryFile(suffix=".grib2") as tmp:
                    tmp.write(gzip.decompress(compressed_file))
                    tmp.flush()
                    ds = xr.load_dataarray(tmp.name, engine="cfgrib", decode_timedelta=True)
    
                    # Drop differing coords
                    for coord in ["valid_time", "step"]:
                        if coord in ds.coords:
                            ds = ds.drop_vars(coord)
    
                    # Attach true timestamp from filename
                    file_time = extract_time_from_filename(f)
                    ds = ds.expand_dims(time=[file_time])
    
                    # Subset
                    subset = SubsetDataRegion(ds, lat_range=lat_range, lon_range=lon_range)
                    datasets.append(subset)
    
            except Exception as e:
                print(f"Failed to load {f}: {e}")
                continue
    
        if datasets:
            day_data = xr.concat(datasets, dim="time", coords="minimal")
            all_datasets.append(day_data)
    
    # ============================================================
    # Combine all dates into one dataset
    # ============================================================
    if all_datasets:
        data_multi = xr.concat(all_datasets, dim="time", coords="minimal")
        print("\nCombined dataset shape:", data_multi.shape)
    else:
        print("No datasets loaded.")


    # ============================================================
    # Saving Data to Single NetCDF Dataset
    # ============================================================
    times = pd.to_datetime(data_multi.time.to_pandas()).tz_localize(None)
    data_multi = data_multi.assign_coords(time=times)
    
    # Now you can save safely
    outputFile = f"MRMSReflectivity_{region}_TRACERregion_{datestrings[0]}-{datestrings[-1]}.nc"
    outputFilePath = os.path.join(outputPath, outputFile)
    data_multi.to_netcdf(outputFilePath)
    return data_multi

In [17]:
########################
#DATA RETRIEVAL

In [ ]:
# ============================================================
# Main Loop: Load multiple dates
# ============================================================
region_options = [
    "CONUS",
    "ALASKA",
    "CARIB",
    "GUAM",
    "HAWAII"
]



product_options = ["MergedReflectivityQC_01.00"]

# Retrieve the user selection from 'Region' 
region = region_options[0]

# Retrieve the user selection from 'MRMS product'
product = product_options[0]

datestrings = ["20220630", "20220701", "20220702"]
lat_range=(25, 35), lon_range=(-100, -90) #subsetting
MRMSData = RetrieveMRMSRadarData(region,product,datestrings,
                                 lat_range, lon_range,
                                 outputPath = "")

In [ ]:
#ADDITIONS FOR DERECHO
# (1) ModelData.region ==> outputFile in place of TRACER
# (2) Use CONUS for TRACER, and HAWAII for Hawaii
# (3) lat_range and lon_range should be based on minmaxes of ModelData